# Measuring a regression model error

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns= "MedHouseVal")
y = 100_000 * housing["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)

y_predicted = model.predict(X_test)

# MAE and RMSE
- The MAE is the average distance between prediction and true value
- The RMSE squares each error, averages, and takes the root to returnt to targets untis

In [2]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

{ "MAE ($):": mean_absolute_error(y_test, y_predicted), "RMSE ($)": root_mean_squared_error(y_test, y_predicted)}

{'MAE ($):': 31105.521289073844, 'RMSE ($)': 46539.919537971495}

In [3]:
import numpy as np
import pandas as pd

true_values = np.array([200,150,300,200,250])

pd.DataFrame({
    name: { "MAE" : mean_absolute_error(true_values, true_values + error), "RMSE": root_mean_squared_error(true_values, true_values + error)}
    for name, error in {
        "every one off by 10": np.array([10, -10, 10, -10, 10]),
        "one off by 50": np.array([0,0,0,0,50]),
        }.items()
    }).T.round(1)

,MAE,RMSE
every one off by 10,10.0,10.0
one off by 50,10.0,22.4


The function a model minimises while it learns is its **loss**.
The measure is judged by afterwawrds is its **metric**.

# Loss vs metric

**Loss** (*pérdida, función de pérdida*): the error the model uses **during training** to improve itself.
- The model makes predictions, measures them with the loss, adjusts itself a tiny bit, and repeats.
- That only works if the loss **changes smoothly**: every tiny improvement must lower it a little.
- So the loss is calculated from smooth numbers (predicted values or probabilities), and it **never jumps**.

The usual loss in regression is the **mean squared error**, where $y_i$ is the true value of row $i$, $\hat{y}_i$ its prediction, and $n$ the number of rows:

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

**Metric** (*métrica de evaluación*): the error **you** use **at the end** to judge the model, usually on validation or test data.
- It's chosen to answer your real question ("how many dollars off are we?", "is this email really spam?").
- It doesn't need to be smooth, because the model doesn't learn from it.

### Do they have to be different?
- **Regression:** predictions are numbers, so nothing jumps. The squared error is good for both learning and judging, so the loss and the metric are often the same family (squared error → RMSE).
- **Classification:** the metric counts right or wrong **decisions** (spam / not spam), and a decision can only flip, so the metric **jumps**.

The loss in classification is the **log loss**, calculated from the predicted probability $p_i$ (with $y_i = 1$ for spam and $0$ for not spam):

$$
\text{log loss} = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\Big]
$$

For one spam email ($y = 1$) it is just $-\log p$, which changes a little whenever $p$ changes a little.

The metrics count decisions. With TP = spam flagged as spam (true positives), FP = normal email flagged as spam (false positives) and FN = spam that was missed (false negatives):

$$
\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN}
$$

TP, FP and FN are counts, so they only change when a decision flips.

Example: the model goes from 40% to 49% spam on a spam email. The log loss improves smoothly ($-\log 0.40 = 0.92$ → $-\log 0.49 = 0.71$), but precision stays frozen until the decision flips at 50%. The model can't learn from a frozen number. So it **trains on the log loss** and is **judged by precision or recall**.

### Why use more than one metric?
Each metric answers a different question, and one number alone can hide things:

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i| \qquad
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
$$

| | MAE | RMSE |
|---|---|---|
| every one off by 10 | (10+10+10+10+10) / 5 = 10.0 | √((100·5) / 5) = 10.0 |
| one off by 50 | 50 / 5 = 10.0 | √(2500 / 5) = 22.4 |

- **MAE:** the typical error. Here both models look the same.
- **RMSE:** squares the errors first, so it punishes big misses. It reveals the single large mistake.

**In one line:** the loss is what the model learns from, and the metrics are what you judge it by.

In [4]:
from sklearn.dummy import DummyRegressor

baselines = {
        "always the mean": DummyRegressor(strategy="mean").fit(X_train, y_train),
        "always the median": DummyRegressor(strategy="median").fit(X_train, y_train),
        }
pd.DataFrame({
    name:{ "MAE($)": mean_absolute_error(y_test, baseline.predict(X_test)),
          "RMSE": root_mean_squared_error(y_test, baseline.predict(X_test))}
    for name, baseline in baselines.items()
        }).T.round(2)

,MAE($),RMSE
always the mean,90606.85,114485.64
always the median,87403.99,117311.67


# Baselines: is the model better than guessing?

To know whether a model is good, we compare it to a **dummy** that ignores the features and always predicts the same number.

### What the median is
**Sort the values and take the one in the middle.**

Five house prices (in thousands):

```
100   150   200   250   1000
            ↑
         median = 200
```

- **Mean** (*media*): add them all and divide → (100+150+200+250+1000) / 5 = **340**

$$
\bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i
$$

- **Median** (*mediana*): the middle value once sorted → **200**. Writing the sorted values as $y_{(1)} \le y_{(2)} \le \dots \le y_{(n)}$:

$$
\text{median} =
\begin{cases}
y_{\left(\frac{n+1}{2}\right)} & \text{if } n \text{ is odd} \\[6pt]
\dfrac{y_{\left(\frac{n}{2}\right)} + y_{\left(\frac{n}{2}+1\right)}}{2} & \text{if } n \text{ is even}
\end{cases}
$$

The one expensive house pulls the mean up to 340, even though 4 of the 5 houses cost 250 or less. The median only cares that the house is on the high side, not how expensive it is, so it stays at 200, a more "typical" price.

With an even number of values, the median is the average of the two in the middle: for `100, 150, 200, 250` it's (150+200)/2 = 175.

### In this data

| | Value |
|---|---|
| Mean house price | 207 195 \$ |
| Median house price | 179 850 \$ |

The mean is higher because some districts are very expensive (808 of them are at the dataset's 500 000 \$ cap), and they pull it up.

### What the two dummies score

| | MAE (\$) | RMSE |
|---|---|---|
| always the mean | 90 607 | **114 486** |
| always the median | **87 404** | 117 312 |

The median wins on MAE and the mean wins on RMSE. That's not luck. If a dummy always predicts the same number $c$:

$$
\text{median} = \arg\min_{c} \sum_{i=1}^{n} |y_i - c|
\qquad
\bar{y} = \arg\min_{c} \sum_{i=1}^{n} (y_i - c)^2
$$

($\arg\min_c$ means "the value of $c$ that makes the sum smallest".)

- The **median** is the single number with the **smallest absolute error (MAE)**.
- The **mean** is the single number with the **smallest squared error (RMSE)**.

So each dummy wins on "its own" metric.

### Why the dummy matters
The real model has **MAE ≈ 31 100 \$**, against **≈ 87 400 \$** for the best dummy. Guessing the same price for every house is almost 3 times worse, so the model really learned something from the features. A model that scored close to the dummy would have learned almost nothing.

In [5]:
from sklearn.metrics import r2_score

{
    "model": r2_score(y_test, y_predicted),
    "always the training mean": r2_score(y_test, baselines["always the mean"].predict(X_test)),
    "model, shifted by $50000": r2_score(y_test, y_predicted + 50_000),
    "model, shifted by $150000": r2_score(y_test, y_predicted + 150_000)
}

{'model': 0.8347108310026643,
 'always the training mean': -0.00021908714592466794,
 'model, shifted by $50000': 0.6379795047054088,
 'model, shifted by $150000': -0.9001639497663978}

# R²: how much better than guessing the mean?

R² compares **the model's error** with **the error of the dummy that always predicts the mean**:

$$
R^2 = 1 - \frac{\text{model's error}}{\text{mean dummy's error}}
= 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}
$$

- $y_i$ = true value, $\hat{y}_i$ = model's prediction, $\bar{y}$ = mean of the true values.
- The differences are **squared**, as in the RMSE.

So R² has the baseline built in. It answers: **"how much smaller is my error than the error of always guessing the mean?"**

### A tiny example
True values `100, 200, 300` (mean = 200):

| | Predictions | Squared errors | Sum |
|---|---|---|---|
| mean dummy | 200, 200, 200 | 100², 0², 100² | 20 000 |
| model | 110, 190, 300 | 10², 10², 0² | 200 |

$$
R^2 = 1 - \frac{200}{20\,000} = 0.99
$$

The model's error is only 1% of the dummy's.

### What the values mean
- **1** → the model never fails (error = 0).
- **0** → no better than always guessing the mean.
- **Negative** → worse than the mean dummy.

R² is **at most 1**, but it is **not between 0 and 1**: it can go as far below 0 as you like.

### In this data

| | R² |
|---|---|
| model | 0.83 |
| always the training mean | ≈ 0 |
| model, shifted by 50 000 \$ | 0.64 |
| model, shifted by 150 000 \$ | −0.90 |

- The **dummy** gets ≈ 0, as expected. It isn't exactly 0 (it's −0.0002) because it learned the mean of the *training* rows, which is slightly different from the mean of the *test* rows.
- **Shifted by 150 000 \$**, the model becomes worse than guessing the mean, so R² is negative.

### Careful: close to 1 doesn't mean "reliable"
- R² has **no units**. 0.83 doesn't tell you how many dollars off the model is. For that you still need the MAE (≈ 31 100 \$).
- **Shifted by 50 000 \$**, the model still gets 0.64, which looks decent, but **every single prediction is 50 000 \$ too high**.

So R² is used **together with** MAE and RMSE, not instead of them: each one answers a different question.

In [6]:
def percentage_measures(y_true, y_pred):
    percentage_errors = np.abs(y_true - y_pred) / np.abs(y_true)
    return {
            "MAPE %": 100 * percentage_errors.mean(),
            "MdAPE %": 100 * np.median(percentage_errors),
            "within 10% %": 100 * (percentage_errors <= 0.10).mean(),
            "within 20% %": 100 * (percentage_errors <= 0.20).mean(),
            }
pd.DataFrame({
        "model": percentage_measures(y_test, y_predicted),
        "always the median": percentage_measures(y_test, baselines["always the median"].predict(X_test))
    }).T.round(1)

,MAPE %,MdAPE %,within 10% %,within 20% %
model,18.0,11.8,43.7,70.4
always the median,53.1,36.2,14.8,28.5


# Percentage errors

Instead of measuring each error in dollars, measure it **as a share of the true price**:

$$
e_i = \frac{|y_i - \hat{y}_i|}{|y_i|}
$$

A house worth 200 000 \$ predicted at 220 000 \$ has $e = 20\,000 / 200\,000 = 0.10$, so it is 10% off.

From those $e_i$:

- **MAPE** (Mean Absolute Percentage Error): the average error, in percent.

$$
\text{MAPE} = \frac{100}{n}\sum_{i=1}^{n} e_i
$$

- **MdAPE** (Median Absolute Percentage Error): the error of a typical house, in percent.

$$
\text{MdAPE} = 100 \cdot \text{median}(e_1, e_2, \dots, e_n)
$$

- **Within 10%**: the share of houses whose prediction is 10% or less away from the true price. $\mathbf{1}[\dots]$ is 1 when the condition is true and 0 when it isn't:

$$
\text{within 10\%} = \frac{100}{n}\sum_{i=1}^{n} \mathbf{1}\big[e_i \le 0.10\big]
$$

  In the code, `percentage_errors <= 0.10` gives `True`/`False` for each house, and `.mean()` of those is the share of `True`.
- **Within 20%**: the same with 0.20.

### In this data

| | MAPE % | MdAPE % | within 10% % | within 20% % |
|---|---|---|---|---|
| model | 18.0 | 11.8 | 43.7 | 70.4 |
| always the median | 53.1 | 36.2 | 14.8 | 28.5 |

### Reading "within 10%" and "within 20%"
"Within 20%" **includes** "within 10%": anything 10% or less away is also 20% or less away. So the model's houses split like this:

| Error | Share of houses |
|---|---|
| 10% or less | 43.7% |
| between 10% and 20% | 70.4 − 43.7 = 26.7% |
| more than 20% | 100 − 70.4 = 29.6% |

### Why MAPE (18%) is bigger than MdAPE (11.8%)
It's the mean vs median idea again. A few predictions have **huge** percentage errors: the worst is 1557% off, probably a very cheap district. A 15 000 \$ house predicted at 250 000 \$ is a moderate miss in dollars but enormous in percent. Those few cases pull the **mean** (MAPE) up. The **median** (MdAPE) ignores how big they are, so it shows that a typical house is about 12% off.

### Another way to compare models
These measures are **in percent**, so they're easy to explain: "the model gets 44% of houses within 10%" means more to most people than "MAE 31 100 \$". Against the median dummy:
- within 10%: **43.7%** vs **14.8%**, about 3 times more houses predicted to within 10%.
- MAPE: **18%** vs **53%**.

**Careful:** percentage errors divide by the true value, so they blow up for very cheap houses (like that 1557%), and they break completely if a true value is 0.

In [7]:
errors = pd.DataFrame({
    "true value": y_test, "absolute error": np.absolute(y_test - y_predicted)
    })
errors["percentage error"] = 100 * errors["absolute error"] / errors["true value"]
errors["price band"] = pd.qcut(errors["true value"], 5, labels=["cheapest fith", "second", "third", "fourht", "most expensive fifth"])
errors.groupby("price band", observed=True)[["absolute error", "percentage error"]].agg(["mean", "median"]).round(0)